# Vertex AI AutoML Tables - Fraud Detection

Trains two AutoML Tabular models and produces **native Vertex AI visualizations**:
ROC curve, PR curve, confusion matrix, and feature importance - all visible in the Vertex AI console without any extra code.

**Two models:**
| Model | Features |
|---|---|
| `automl-fraud-baseline` | Raw tabular features only |
| `automl-fraud-graph-enhanced` | Tabular + 22 Neo4j graph features |

**After running this notebook** (training takes ~1-2 hours per model):
Vertex AI > Model Registry > select model > Evaluate tab

In [ ]:
import sys, subprocess

# Fix NumPy/PyArrow conflict common in Colab Enterprise
# (newer PyArrow in ~/.local clashes with conda NumPy 1.x)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'numpy<2',
    'pyarrow>=12,<15',
    'scikit-learn>=1.3',
    'lightgbm>=4.0',
    'gcsfs',
    'google-cloud-aiplatform',
], check=True)

print("Packages ready. Restarting kernel to apply changes...")

# Restart kernel - re-run all cells after this completes
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
import subprocess
PROJECT_ID   = subprocess.run(['gcloud','config','get-value','project'],
                               capture_output=True, text=True).stdout.strip()
LOCATION     = 'us-east1'
GCS_BUCKET   = 'neo4j-fraud-detection'
DATASET_PATH = f'gs://{GCS_BUCKET}/artifacts/demo_dataset.parquet'
AUTOML_GCS   = f'gs://{GCS_BUCKET}/automl'

BUDGET_MILLI_NODE_HOURS = 1000   # 1 node-hour minimum (~1-2 hrs real time, ~$4-8)

print(f'Project  : {PROJECT_ID}')
print(f'Location : {LOCATION}')
print(f'AutoML data will be uploaded to: {AUTOML_GCS}/')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy  as np
from sklearn.preprocessing import LabelEncoder
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION)
print('google-cloud-aiplatform ready.')

---
## 1. Load and Prepare Datasets

In [ ]:
print('Loading merged dataset ...')
df = pd.read_parquet(DATASET_PATH)
print(f'Loaded: {df.shape}')

# Columns to exclude from features
EXCLUDE = {'TransactionID', 'TransactionDT'}   # not predictive identifiers
EMB_COLS = [f'emb_{i}' for i in range(64)]     # exclude FastRP embeddings from AutoML
                                                # (not interpretable as feature names)

GRAPH_COLS = [c for c in [
    'card_tx_count','card_fraud_count','card_fraud_rate',
    'payer_email_tx_count','payer_email_fraud_count','payer_email_fraud_rate',
    'recip_email_tx_count','recip_email_fraud_count','recip_email_fraud_rate',
    'billing_tx_count','billing_fraud_count','billing_fraud_rate',
    'device_tx_count','device_fraud_count','device_fraud_rate',
    'os_browser_tx_count','os_browser_fraud_count','os_browser_fraud_rate',
    'is_proxy','proxy_fraud_rate','prev_card_is_fraud','prev_card_dt_gap',
] if c in df.columns]

# Temporal train/test split column
SPLIT_DT = 12_528_000
df['data_split'] = df['TransactionDT'].apply(lambda x: 'TRAIN' if x <= SPLIT_DT else 'TEST')
print(f'Train: {(df["data_split"]=="TRAIN").sum():,}  Test: {(df["data_split"]=="TEST").sum():,}')
print(f'Graph feature columns: {len(GRAPH_COLS)}')

---
## 2. Export CSVs to GCS

AutoML Tables requires CSV (or BigQuery) input.

- **Tabular-only CSV**: raw transaction features + target
- **Graph-enhanced CSV**: same + 22 Neo4j graph features

Embedding columns (`emb_0`...`emb_63`) are excluded - they're not interpretable as feature names in the AutoML feature importance chart.

In [ ]:
def export_to_gcs(df, keep_cols, gcs_path, name):
    local = f'/tmp/{name}.csv'
    print(f'Exporting {name}: {len(keep_cols)} columns ...')
    df[keep_cols].to_csv(local, index=False)
    import subprocess, os
    size_mb = os.path.getsize(local) / 1e6
    print(f'  Local file: {size_mb:.0f} MB')
    result = subprocess.run(['gsutil','-m','cp', local, gcs_path],
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(f'  ERROR: {result.stderr}')
    else:
        print(f'  Uploaded -> {gcs_path}')
    return gcs_path

# Columns for each experiment
TAB_COLS = [c for c in df.columns
            if c not in EXCLUDE and c not in GRAPH_COLS and c not in EMB_COLS
            and c != 'data_split']

tabular_cols       = TAB_COLS + ['isFraud', 'data_split']
graph_enhanced_cols = TAB_COLS + GRAPH_COLS + ['isFraud', 'data_split']

print(f'Tabular-only cols   : {len(tabular_cols)-2} features + isFraud + data_split')
print(f'Graph-enhanced cols : {len(graph_enhanced_cols)-2} features + isFraud + data_split')

In [ ]:
gcs_tabular = export_to_gcs(
    df, tabular_cols,
    f'{AUTOML_GCS}/tabular_only.csv',
    'tabular_only'
)

gcs_graph = export_to_gcs(
    df, graph_enhanced_cols,
    f'{AUTOML_GCS}/graph_enhanced.csv',
    'graph_enhanced'
)

---
## 3. Create Vertex AI Datasets

In [ ]:
print('Creating Vertex AI Dataset: tabular-only ...')
dataset_tabular = aiplatform.TabularDataset.create(
    display_name = 'fraud-tabular-only',
    gcs_source   = gcs_tabular,
)
print(f'  Resource: {dataset_tabular.resource_name}')

print('Creating Vertex AI Dataset: graph-enhanced ...')
dataset_graph = aiplatform.TabularDataset.create(
    display_name = 'fraud-graph-enhanced',
    gcs_source   = gcs_graph,
)
print(f'  Resource: {dataset_graph.resource_name}')
print('\nDatasets ready.')

---
## 4. Launch AutoML Training Jobs

Training runs asynchronously (`sync=False`). Each job takes approximately **1-2 hours**.

Monitor progress at: Vertex AI > Training > Custom Jobs

In [ ]:
def make_automl_job(display_name):
    return aiplatform.AutoMLTabularTrainingJob(
        display_name                  = display_name,
        optimization_prediction_type  = 'classification',
        optimization_objective        = 'maximize-au-prc',   # PR-AUC: right metric for imbalanced fraud
    )

# ── Experiment 1: Tabular Baseline ────────────────────────────────────────────
print('Launching AutoML job 1: tabular baseline ...')
job_tabular = make_automl_job('automl-fraud-baseline')
model_tabular = job_tabular.run(
    dataset                    = dataset_tabular,
    target_column              = 'isFraud',
    predefined_split_column_name = 'data_split',  # use our temporal split
    budget_milli_node_hours    = BUDGET_MILLI_NODE_HOURS,
    model_display_name         = 'automl-fraud-baseline',
    sync                       = False,           # don't block - runs in background
)
print(f'  Job submitted: {job_tabular.resource_name}')

In [ ]:
# ── Experiment 2: Graph-Enhanced ─────────────────────────────────────────────
print('Launching AutoML job 2: graph-enhanced ...')
job_graph = make_automl_job('automl-fraud-graph-enhanced')
model_graph = job_graph.run(
    dataset                    = dataset_graph,
    target_column              = 'isFraud',
    predefined_split_column_name = 'data_split',
    budget_milli_node_hours    = BUDGET_MILLI_NODE_HOURS,
    model_display_name         = 'automl-fraud-graph-enhanced',
    sync                       = False,
)
print(f'  Job submitted: {job_graph.resource_name}')

---
## 5. Monitor and View Results

Once training completes (~1-2 hours), go to the Vertex AI console to see:

### Training progress
**Vertex AI > Training** - shows both running jobs with estimated completion time

### Model evaluation (confusion matrix, ROC, feature importance)
**Vertex AI > Model Registry** > select model > **Evaluate** tab

You will see:
- **Confusion matrix** at adjustable thresholds
- **ROC curve** with AUC
- **PR curve** with AUC (most relevant for fraud - imbalanced)
- **Feature importance** bar chart - look for `card_fraud_rate`, `prev_card_is_fraud`, `device_fraud_rate` in the graph-enhanced model

### Compare the two models
Open both models in the Evaluate tab side by side and compare PR-AUC.

In [ ]:
# Print monitoring links
print('Monitor training jobs:')
print(f'  https://console.cloud.google.com/vertex-ai/training/training-pipelines?project={PROJECT_ID}')
print()
print('View models after training:')
print(f'  https://console.cloud.google.com/vertex-ai/models?project={PROJECT_ID}')
print()
print('Job resource names (save these):')
print(f'  Tabular baseline : {job_tabular.resource_name}')
print(f'  Graph-enhanced   : {job_graph.resource_name}')